In [ ]:
import torch
import torchvision

print(torch.__version__)
print(torchvision.__version__)
print(torch.cuda.is_available())

In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# -------------------------
# 1. Paths
# -------------------------
train_dir = r"C:\Users\A7\Downloads\archive\Skin cancer ISIC The International Skin Imaging Collaboration\Train"
test_dir  = r"C:\Users\A7\Downloads\archive\Skin cancer ISIC The International Skin Imaging Collaboration\Test"


In [ ]:
from torchvision import transforms

train_transforms_resnet = transforms.Compose([
    transforms.Resize((224, 224)),

    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),

    transforms.RandomRotation(20),

    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transforms_resnet = transforms.Compose([
    transforms.Resize((224, 224)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
#train_loader 

# -------------------------
# 3. Load datasets
# -------------------------


# it works like this when image classification is present in form of folders
train_dataset = datasets.ImageFolder(root=train_dir, transform=train_transforms_resnet)
test_dataset  = datasets.ImageFolder(root=test_dir, transform=test_transforms_resnet)






#splitting the data into validation and training 
from torch.utils.data import random_split
val_ratio = 0.2

train_size = int((1 - val_ratio) * len(train_dataset))
val_size = len(train_dataset) - train_size

train_data, val_data = random_split(train_dataset, [train_size, val_size])

val_data.dataset.transform = test_transforms_resnet




# -------------------------
# 4. Create DataLoaders
# -------------------------
train_loader = DataLoader(
    train_data,
    batch_size=32,
    shuffle=True,
    num_workers=2
)

val_loader = DataLoader(
    val_data,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

# -------------------------
# 5. Check
# -------------------------
print("Classes:", train_dataset.classes)
print("Train samples:", len(train_data))
print("Val samples:", len(val_data))
print("Test samples:", len(test_dataset))

In [ ]:
# USING RESNET 50 MODEL

import torch
import torch.nn as nn
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

modelnet = models.resnet50(weights='IMAGENET1K_V1')

# Replace final layer
num_features = modelnet.fc.in_features

modelnet.fc = nn.Linear(num_features, 9)

modelnet = modelnet.to(device)

In [ ]:
# 4. Optimizer

# Use lower learning rate:
import torch.optim as optim
optimizer = torch.optim.AdamW(modelnet.parameters(), lr=1e-4, weight_decay=1e-4)

In [ ]:
# 3. Weighted loss (VERY IMPORTANT)

# Your dataset is imbalanced.

# Use weighted loss:
from collections import Counter

class_counts = Counter(train_dataset.targets)
print(class_counts)

total_samples = sum(class_counts.values())

weights = []

for i in range(len(train_dataset.classes)):
    weights.append(total_samples / class_counts[i])

weights = torch.tensor(weights, dtype=torch.float).to(device)

criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.1)

In [ ]:
epochs = 25
best_acc = 0.0

# -------------------------
# Scheduler (ADD THIS)
# -------------------------
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=epochs
)

for epoch in range(epochs):

    # ---------------- TRAIN ----------------
    modelnet.train()
    train_loss = 0.0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = modelnet(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss = train_loss / len(train_loader)

    # ---------------- VALIDATION ----------------
    modelnet.eval()
    val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = modelnet(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()

            _, preds = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (preds == labels).sum().item()

    val_loss = val_loss / len(val_loader)
    val_acc = 100 * correct / total

    # ---------------- PRINT ----------------
    print(f"Epoch [{epoch+1}/{epochs}] "
          f"Train Loss: {train_loss:.4f} "
          f"Val Loss: {val_loss:.4f} "
          f"Val Acc: {val_acc:.2f}%")

    # ---------------- SAVE BEST MODEL ----------------
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(modelnet.state_dict(), "best_model.pth")

    # ---------------- SCHEDULER STEP (ADD THIS) ----------------
    scheduler.step()

In [ ]:
modelnet.eval()

correct = 0
total = 0

val_loss = 0.0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = modelnet(images)

        loss = criterion(outputs, labels)

        val_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total

print(f"Testing Loss: {val_loss/len(test_loader):.4f}")
print(f" Testing Accuracy: {accuracy:.2f}%")

In [ ]:
images, label = next(iter(test_loader))
img = images[11]
img = img.unsqueeze(0)
print(img.shape)
label[11]

In [ ]:
output = modelnet(img)
print(output)

pred = output.argmax(dim=1)

print(pred)

In [ ]:
#WORKING ON CONFUSION MATRIX
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

all_preds = []
all_labels = []

modelnet.eval()

with torch.no_grad():
    for images, labels in test_loader:
        # images = images.to(device)
        # labels = labels.to(device)

        outputs = modelnet(images)

        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

In [ ]:
cm = confusion_matrix(all_labels, all_preds)

print(cm)

In [ ]:
plt.figure(figsize=(8,6))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=train_dataset.classes,
    yticklabels=train_dataset.classes
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")

plt.show()

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(
    all_labels,
    all_preds,
    target_names=train_dataset.classes
))